# Agentic RAG với Deep Agents — Pháp Lý Việt Nam

Triển khai **Agentic RAG** cho tra cứu văn bản pháp luật Việt Nam sử dụng framework Deep Agents với kiến trúc đa sub-agent. Orchestrator điều phối hai chuyên gia:
- **retrieval-specialist** — tra cứu vector DB nội bộ và đánh giá độ liên quan
- **web-researcher** — tìm kiếm thông tin pháp lý bên ngoài khi cần thiết

Pipeline đảm bảo câu trả lời luôn có căn cứ từ tài liệu, không phụ thuộc vào kiến thức nội tại của model.

In [1]:
%pip install -Uq deepagents

Note: you may need to restart the kernel to use updated packages.


## Bước 1 — Cài Đặt & Imports

Cài `deepagents` và import các thư viện cần thiết: LangChain tools, HuggingFace embeddings, và các lớp Deep Agents.

In [2]:
import os
import math
import json
from typing import List, Dict, Any
from langchain_core.tools import tool
from langchain_huggingface import HuggingFaceEmbeddings
from deepagents import create_deep_agent
from deepagents.middleware.subagents import SubAgent

## Bước 2 — Vector Store In-Memory

`TinyVectorDB` mã hóa tài liệu pháp lý bằng `HuggingFaceEmbeddings` (BAAI/bge-m3). Khác với các notebook khác, mỗi tài liệu được lưu kèm `id` và `metadata` (nguồn văn bản, chủ đề) để hỗ trợ truy xuất có cấu trúc.

In [3]:
def cosine_similarity(v1, v2) -> float:
    dot = sum(x * y for x, y in zip(v1, v2))
    mag1 = math.sqrt(sum(x * x for x in v1))
    mag2 = math.sqrt(sum(x * x for x in v2))

    if mag1 == 0 or mag2 == 0:
        return 0.0

    return dot / (mag1 * mag2)


class TinyVectorDB:
    def __init__(self):
        self.docs = []
        self.encoder = HuggingFaceEmbeddings(
            model_name="BAAI/bge-m3",
            encode_kwargs={"normalize_embeddings": True},
        )

    def add_documents(self, documents: List[Dict[str, Any]]):
        texts = [doc["text"] for doc in documents]
        embeddings = self.encoder.embed_documents(texts)

        for doc, emb in zip(documents, embeddings):
            self.docs.append(
                {
                    "id": doc.get("id"),
                    "text": doc["text"],
                    "metadata": doc.get("metadata", {}),
                    "embedding": emb,
                }
            )

    def search(self, query: str, top_k: int = 4):
        query_embedding = self.encoder.embed_query(query)

        results = []
        for doc in self.docs:
            score = cosine_similarity(query_embedding, doc["embedding"])
            results.append(
                {
                    "id": doc["id"],
                    "text": doc["text"],
                    "metadata": doc["metadata"],
                    "score": round(score, 4),
                }
            )

        results.sort(key=lambda x: x["score"], reverse=True)
        return results[:top_k]


vector_db = TinyVectorDB()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

## Bước 3 — Công Cụ Agent

Ba `@tool` trang bị cho các sub-agent khả năng truy xuất và đánh giá:
- **`internal_vector_search`** — tìm kiếm vector DB nội bộ; luôn được gọi trước
- **`document_relevance_grader`** — chấm điểm mức độ liên quan của ngữ cảnh đã truy xuất bằng token overlap; quyết định có cần dùng web không
- **`web_search`** — dự phòng qua Tavily API (mô phỏng nếu không có API key)

In [ ]:
@tool
def internal_vector_search(query: str, top_k: int = 4) -> str:
    """
    Tìm kiếm trong vector database nội bộ về văn bản pháp luật Việt Nam.
    Luôn dùng công cụ này trước khi tìm kiếm web.
    """
    results = vector_db.search(query=query, top_k=top_k)

    return json.dumps(
        {
            "tool": "internal_vector_search",
            "query": query,
            "results": results,
        },
        ensure_ascii=False,
        indent=2,
    )

In [ ]:
@tool
def document_relevance_grader(query: str, retrieved_context: str) -> str:
    """
    Đánh giá xem ngữ cảnh nội bộ đã truy xuất có đủ để trả lời câu hỏi không.
    """
    query_tokens = set(query.lower().split())
    context_tokens = set(retrieved_context.lower().split())

    overlap = len(query_tokens.intersection(context_tokens))

    if overlap >= 3:
        result = {
            "relevant": "yes",
            "confidence": "high",
            "should_use_web": False,
            "reason": "Ngữ cảnh nội bộ có vẻ đầy đủ.",
        }
    elif overlap >= 1:
        result = {
            "relevant": "partial",
            "confidence": "medium",
            "should_use_web": True,
            "reason": "Ngữ cảnh nội bộ liên quan nhưng chưa đầy đủ.",
        }
    else:
        result = {
            "relevant": "no",
            "confidence": "low",
            "should_use_web": True,
            "reason": "Ngữ cảnh nội bộ không đủ thông tin.",
        }

    return json.dumps(
        {
            "tool": "document_relevance_grader",
            "query": query,
            **result,
        },
        ensure_ascii=False,
        indent=2,
    )

In [ ]:
@tool
def web_search(query: str, max_results: int = 5) -> str:
    """
    Tìm kiếm web dự phòng khi truy xuất nội bộ không đủ thông tin.
    """
    api_key = os.environ.get("TAVILY_API_KEY")

    if not api_key:
        return json.dumps(
            {
                "tool": "web_search",
                "query": query,
                "warning": "Không tìm thấy TAVILY_API_KEY. Đây là kết quả web mô phỏng.",
                "results": [
                    {
                        "title": "Kết quả mô phỏng",
                        "content": "Tìm kiếm web bên ngoài chưa được thực thi.",
                        "url": None,
                    }
                ],
            },
            ensure_ascii=False,
            indent=2,
        )

    from tavily import TavilyClient

    client = TavilyClient(api_key=api_key)
    response = client.search(
        query=query,
        max_results=max_results,
        include_raw_content=False,
    )

    return json.dumps(
        {
            "tool": "web_search",
            "query": query,
            "results": response.get("results", []),
        },
        ensure_ascii=False,
        indent=2,
    )

## Bước 4 — System Prompt của Orchestrator

`AGENTIC_RAG_PROMPT` định nghĩa chính sách xử lý cho orchestrator: ưu tiên tra cứu nội bộ, ủy quyền cho sub-agent phù hợp, và yêu cầu câu trả lời phải trích dẫn điều khoản cụ thể.

In [ ]:
AGENTIC_RAG_PROMPT = """
Bạn là Deep Agent Agentic RAG chuyên về pháp luật Việt Nam.

Chính sách xử lý:

1. Với mỗi câu hỏi, hãy ưu tiên tra cứu cơ sở tri thức nội bộ trước.
2. Ưu tiên giao việc truy xuất nội bộ cho sub-agent chuyên truy xuất (retrieval-specialist).
3. Nếu bằng chứng nội bộ không đầy đủ, yếu hoặc lỗi thời, hãy giao cho sub-agent nghiên cứu web (web-researcher).
4. Sử dụng bộ nhớ/filesystem cho các tác vụ nhiều bước.
5. Câu trả lời cuối phải bao gồm:
   - câu trả lời trực tiếp
   - bằng chứng nội bộ đã sử dụng (trích dẫn điều khoản cụ thể)
   - bằng chứng web đã sử dụng, nếu có
   - sự không chắc chắn hoặc thông tin còn thiếu, nếu có
6. Không bao giờ trả lời chỉ từ bộ nhớ của model.
7. Trả lời bằng tiếng Việt.
"""

## Bước 5 — Sub-agents & Khởi Tạo Deep Agent

Hai sub-agent được cấu hình với công cụ riêng biệt, sau đó được đăng ký vào orchestrator qua `create_deep_agent`:
- **`retrieval-specialist`** — sở hữu `internal_vector_search` + `document_relevance_grader`
- **`web-researcher`** — sở hữu `web_search`

Orchestrator có quyền gọi trực tiếp tất cả công cụ lẫn ủy quyền cho sub-agent tùy tình huống.

In [ ]:
retrieval_specialist: SubAgent = {
    "name": "retrieval-specialist",
    "description": "Tìm kiếm vector DB nội bộ về văn bản pháp luật Việt Nam và đánh giá mức độ liên quan.",
    "system_prompt": """
Bạn là chuyên gia truy xuất văn bản pháp lý.

Gọi internal_vector_search để tìm kiếm tài liệu pháp luật,
kiểm tra tài liệu được trả về,
sau đó gọi document_relevance_grader để đánh giá mức độ liên quan.
""",
    "tools": [
        internal_vector_search,
        document_relevance_grader,
    ],
}

web_researcher: SubAgent = {
    "name": "web-researcher",
    "description": "Tìm kiếm thông tin pháp lý bên ngoài khi bằng chứng nội bộ không đủ.",
    "system_prompt": """
Bạn là chuyên gia nghiên cứu pháp lý trực tuyến.

Chỉ dùng web_search khi bằng chứng nội bộ yếu hoặc không có.
""",
    "tools": [
        web_search,
    ],
}

agent = create_deep_agent(
    model="openai:gpt-4o-mini",
    tools=[
        internal_vector_search,
        document_relevance_grader,
        web_search,
    ],
    system_prompt=AGENTIC_RAG_PROMPT,
    subagents=[
        retrieval_specialist,
        web_researcher,
    ],
)

## Bước 6 — Cơ Sở Tri Thức

Nạp bốn văn bản pháp lý Việt Nam vào vector DB với metadata đầy đủ: Bộ luật Lao động 2019 (thời giờ làm việc, làm thêm giờ), Luật Doanh nghiệp 2020 (công ty cổ phần), Luật Đất đai 2013 (quyền sử dụng đất).

In [ ]:
def load_sample_documents():
    vector_db.add_documents(
        [
            {
                "id": "lao_dong_001",
                "text": (
                    "Theo Điều 105 Bộ luật Lao động 2019, thời giờ làm việc bình thường "
                    "không quá 8 giờ trong một ngày và không quá 48 giờ trong một tuần."
                ),
                "metadata": {
                    "source": "Bo_luat_Lao_dong_2019",
                    "topic": "thoi_gio_lam_viec",
                },
            },
            {
                "id": "lao_dong_002",
                "text": (
                    "Điều 107 Bộ luật Lao động 2019 quy định người lao động làm thêm giờ "
                    "không được vượt quá 50% số giờ làm việc bình thường trong ngày; "
                    "tổng số giờ làm việc và làm thêm không quá 12 giờ trong một ngày."
                ),
                "metadata": {
                    "source": "Bo_luat_Lao_dong_2019",
                    "topic": "lam_them_gio",
                },
            },
            {
                "id": "doanh_nghiep_001",
                "text": (
                    "Điều 111 Luật Doanh nghiệp 2020 quy định công ty cổ phần là doanh nghiệp "
                    "có vốn điều lệ được chia thành nhiều phần bằng nhau gọi là cổ phần; "
                    "số lượng cổ đông tối thiểu là 03 và không hạn chế số lượng tối đa."
                ),
                "metadata": {
                    "source": "Luat_Doanh_nghiep_2020",
                    "topic": "cong_ty_co_phan",
                },
            },
            {
                "id": "dat_dai_001",
                "text": (
                    "Khoản 1 Điều 166 Luật Đất đai 2013 quy định người sử dụng đất có quyền "
                    "được cấp Giấy chứng nhận quyền sử dụng đất, quyền sở hữu nhà ở và "
                    "tài sản khác gắn liền với đất."
                ),
                "metadata": {
                    "source": "Luat_Dat_dai_2013",
                    "topic": "quyen_su_dung_dat",
                },
            },
        ]
    )

In [10]:
def ask(question: str):
    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": question,
                }
            ]
        }
    )

    return result["messages"][-1].content

## Bước 7 — Chạy Demo

Nạp dữ liệu và đặt câu hỏi so sánh Agentic RAG với RAG thông thường trong bối cảnh pháp lý. Orchestrator sẽ ủy quyền cho `retrieval-specialist` trước, sau đó tổng hợp câu trả lời có trích dẫn nguồn.

In [ ]:
load_sample_documents()

question = "Giải thích Agentic RAG và so sánh với RAG thông thường trong bối cảnh tra cứu văn bản pháp luật Việt Nam."
answer = ask(question)

In [ ]:
print(answer[0])